In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, train_test_split
from tabulate import tabulate
import pickle
from data_preprocessing import load_and_preprocess_data
from training import train_and_evaluate_fold
from utils import ensure_directory

def main(csv_path, output_mode='device', n_splits=1, target_labels=None, target_traffic_types=None, distillation_method='standard'):
    """Main function to orchestrate data loading, preprocessing, training, and evaluation."""
    output_dir = Path("results")
    checkpoint_dir = Path("checkpoints")
    ensure_directory(output_dir)
    ensure_directory(checkpoint_dir)

    dataset_name = Path(csv_path).stem
    print(f"\nProcessing dataset: {dataset_name} with output mode: {output_mode}")

    # Load and preprocess data
    try:
        X, y, le, feature_names, scaler, num_classes_device, num_classes_attack, output_dir, checkpoint_dir = load_and_preprocess_data(
            csv_path, target_labels=target_labels, target_traffic_types=target_traffic_types,
            output_mode=output_mode, distillation_method=distillation_method)
        if X is None or y is None:
            print(f"Error: Failed to load or preprocess data from {csv_path}")
            return
    except Exception as e:
        print(f"Error loading/preprocessing data: {e}")
        return

    # Save feature names and scaler
    with open(checkpoint_dir / f"{dataset_name}_feature_names.pkl", 'wb') as f:
        pickle.dump(feature_names, f)
    with open(checkpoint_dir / f"{dataset_name}_scaler.pkl", 'wb') as f:
        pickle.dump(scaler, f)

    all_results = []
    if n_splits > 1:
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
        if output_mode == 'multi':
            y_for_stratify = np.column_stack(y)
        else:
            y_for_stratify = y

        for fold, (train_idx, test_idx) in enumerate(skf.split(X, y_for_stratify[:, 0] if output_mode == 'multi' else y)):
            print(f"\nProcessing Fold {fold + 1}/{n_splits}")
            X_train_val = X[train_idx]
            X_test = X[test_idx]
            if output_mode == 'multi':
                y_train_val = (y[0][train_idx], y[1][train_idx])
                y_test = (y[0][test_idx], y[1][test_idx])
            else:
                y_train_val = y[train_idx]
                y_test = y[test_idx]

            fold_results = train_and_evaluate_fold(
                X_train_val, y_train_val, X_test, y_test, feature_names, dataset_name,
                output_dir, checkpoint_dir, output_mode, le, num_classes_device, num_classes_attack,
                n_splits, fold=fold + 1, distillation_method=distillation_method)
            all_results.extend(fold_results)
    else:
        if output_mode == 'multi':
            X_train_val, X_test, y_train_val, y_test = train_test_split(
                X, y, test_size=0.2, random_state=42, stratify=np.column_stack(y))
            y_train_val = (y_train_val[0], y_train_val[1])
            y_test = (y_test[0], y_test[1])
        else:
            X_train_val, X_test, y_train_val, y_test = train_test_split(
                X, y, test_size=0.2, random_state=42, stratify=y)

        fold_results = train_and_evaluate_fold(
            X_train_val, y_train_val, X_test, y_test, feature_names, dataset_name,
            output_dir, checkpoint_dir, output_mode, le, num_classes_device, num_classes_attack,
            n_splits=1, fold=None, distillation_method=distillation_method)
        all_results.extend(fold_results)

    # Aggregate and save results
    results_df = pd.DataFrame(all_results)
    results_csv_path = output_dir / f"{dataset_name}_results.csv"
    results_df.to_csv(results_csv_path, index=False)
    print(f"\nSaved all results to {results_csv_path}")

    # Generate summary plots
    try:
        plt.figure(figsize=(12, 6))
        sns.boxplot(x='Model', y='Accuracy', data=results_df)
        plt.title(f"Model Accuracy Comparison ({dataset_name})")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.savefig(output_dir / f"{dataset_name}_accuracy_boxplot.pdf", format='pdf', dpi=300)
        plt.close()

        plt.figure(figsize=(12, 6))
        sns.boxplot(x='Model', y='Train Time (s)', data=results_df)
        plt.title(f"Model Training Time Comparison ({dataset_name})")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.savefig(output_dir / f"{dataset_name}_train_time_boxplot.pdf", format='pdf', dpi=300)
        plt.close()

        plt.figure(figsize=(12, 6))
        sns.boxplot(x='Model', y='Num Parameters', data=results_df)
        plt.title(f"Model Complexity Comparison ({dataset_name})")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.savefig(output_dir / f"{dataset_name}_parameters_boxplot.pdf", format='pdf', dpi=300)
        plt.close()
    except Exception as e:
        print(f"Error generating summary plots: {e}")

    # Print average metrics
    print("\nAverage Metrics Across Folds:")
    avg_metrics = results_df.groupby('Model').mean(numeric_only=True)
    print(tabulate(avg_metrics.reset_index(), headers='keys', tablefmt='grid', floatfmt=".4f"))

if __name__ == "__main__":
    csv_path = r"C:\Users\danie\Downloads\IoMT2024_All_data_merged.csv"
    ##       parser.add_argument("--distillation_method", 
    #type=str, choices=['standard', 'active', 'feature_matching', 'gradient_matching', 'combined', 'coreset', 'generative'], 
    #default='standard', help="Distillation method")

    main(csv_path, output_mode='traffic', n_splits=1, target_labels=None, target_traffic_types=None, distillation_method='standard')


Processing dataset: IoMT2024_All_data_merged with output mode: traffic
Loaded IoMT2024_All_data_merged.csv with 202322 rows and 73 columns.

Replaced NaN/inf with median values.

Null values per column after preprocessing:
Entropy            0
Burst_count        0
TCP_response       0
TCP_mss_values     0
STUN_method        0
                  ..
IP_Dst_Entropy     0
Payload_Len_Q1     0
Payload_Len_Q3     0
Payload_Len_IQR    0
Traffic Type       0
Length: 73, dtype: int64

Number of duplicate rows: 1
Dropped 1 duplicate rows.

Column data types:
Entropy            float64
Burst_count          int64
TCP_response         int64
TCP_mss_values       int64
STUN_method          int64
                    ...   
IP_Dst_Entropy     float64
Payload_Len_Q1       int64
Payload_Len_Q3       int64
Payload_Len_IQR      int64
Traffic Type        object
Length: 73, dtype: object

NaN values in numeric columns before imputation:
Series([], dtype: int64)

Traffic Type class distribution after sampling

  0%|          | 0/200 [00:00<?, ?it/s]

C:\Users\danie\anaconda3\envs\HybridDualDistillation\lib\site-packages\shap\utils\_clustering.py:170: UserWarning: No/low signal found from feature 3 (this is typically caused by constant or near-constant features)! Cluster distances can't be computed for it (so setting all redundancy distances to 1).
  warnings.warn(
C:\Users\danie\anaconda3\envs\HybridDualDistillation\lib\site-packages\shap\utils\_clustering.py:170: UserWarning: No/low signal found from feature 4 (this is typically caused by constant or near-constant features)! Cluster distances can't be computed for it (so setting all redundancy distances to 1).
  warnings.warn(
C:\Users\danie\anaconda3\envs\HybridDualDistillation\lib\site-packages\shap\utils\_clustering.py:170: UserWarning: No/low signal found from feature 5 (this is typically caused by constant or near-constant features)! Cluster distances can't be computed for it (so setting all redundancy distances to 1).
  warnings.warn(
C:\Users\danie\anaconda3\envs\HybridDual

Saved SHAP summary plot for Traffic class ARP_Spoofing to D:\HybridDualDistillation\IoMTDataset\IoMT-Distillation\checkpoints\shap_summary_teacher_1d_cnn_traffic_arp_spoofing_fold_single.pdf
Saved SHAP bar plot for Traffic class ARP_Spoofing to D:\HybridDualDistillation\IoMTDataset\IoMT-Distillation\checkpoints\shap_bar_teacher_1d_cnn_traffic_arp_spoofing_fold_single.pdf
Saved SHAP beeswarm plot for Traffic class ARP_Spoofing to D:\HybridDualDistillation\IoMTDataset\IoMT-Distillation\checkpoints\shap_beeswarm_teacher_1d_cnn_traffic_arp_spoofing_fold_single.pdf
Saved SHAP local bar plot for Traffic class ARP_Spoofing to D:\HybridDualDistillation\IoMTDataset\IoMT-Distillation\checkpoints\shap_local_bar_teacher_1d_cnn_traffic_arp_spoofing_fold_single.pdf
Saved SHAP clustered bar plot for Traffic class ARP_Spoofing to D:\HybridDualDistillation\IoMTDataset\IoMT-Distillation\checkpoints\shap_clustered_bar_teacher_1d_cnn_traffic_arp_spoofing_fold_single.pdf
Saved SHAP partitioned bar plot for

  0%|          | 0/200 [00:00<?, ?it/s]

...

C:\Users\danie\anaconda3\envs\HybridDualDistillation\lib\site-packages\shap\utils\_clustering.py:170: UserWarning: No/low signal found from feature 4 (this is typically caused by constant or near-constant features)! Cluster distances can't be computed for it (so setting all redundancy distances to 1).
  warnings.warn(
C:\Users\danie\anaconda3\envs\HybridDualDistillation\lib\site-packages\shap\utils\_clustering.py:170: UserWarning: No/low signal found from feature 5 (this is typically caused by constant or near-constant features)! Cluster distances can't be computed for it (so setting all redundancy distances to 1).
  warnings.warn(
C:\Users\danie\anaconda3\envs\HybridDualDistillation\lib\site-packages\shap\utils\_clustering.py:170: UserWarning: No/low signal found from feature 6 (this is typically caused by constant or near-constant features)! Cluster distances can't be computed for it (so setting all redundancy distances to 1).
  warnings.warn(
 28%|███████████████████████████████████

Saved SHAP summary plot for Traffic class ARP_Spoofing to D:\HybridDualDistillation\IoMTDataset\IoMT-Distillation\checkpoints\shap_summary_student_1d_cnn_kd_cosine_combined_traffic_arp_spoofing_fold_single.pdf
Saved SHAP bar plot for Traffic class ARP_Spoofing to D:\HybridDualDistillation\IoMTDataset\IoMT-Distillation\checkpoints\shap_bar_student_1d_cnn_kd_cosine_combined_traffic_arp_spoofing_fold_single.pdf
Saved SHAP beeswarm plot for Traffic class ARP_Spoofing to D:\HybridDualDistillation\IoMTDataset\IoMT-Distillation\checkpoints\shap_beeswarm_student_1d_cnn_kd_cosine_combined_traffic_arp_spoofing_fold_single.pdf
Saved SHAP local bar plot for Traffic class ARP_Spoofing to D:\HybridDualDistillation\IoMTDataset\IoMT-Distillation\checkpoints\shap_local_bar_student_1d_cnn_kd_cosine_combined_traffic_arp_spoofing_fold_single.pdf
Saved SHAP clustered bar plot for Traffic class ARP_Spoofing to D:\HybridDualDistillation\IoMTDataset\IoMT-Distillation\checkpoints\shap_clustered_bar_student_1d_

  0%|          | 0/200 [00:00<?, ?it/s]

C:\Users\danie\anaconda3\envs\HybridDualDistillation\lib\site-packages\shap\utils\_clustering.py:170: UserWarning: No/low signal found from feature 3 (this is typically caused by constant or near-constant features)! Cluster distances can't be computed for it (so setting all redundancy distances to 1).
  warnings.warn(
C:\Users\danie\anaconda3\envs\HybridDualDistillation\lib\site-packages\shap\utils\_clustering.py:170: UserWarning: No/low signal found from feature 4 (this is typically caused by constant or near-constant features)! Cluster distances can't be computed for it (so setting all redundancy distances to 1).
  warnings.warn(
C:\Users\danie\anaconda3\envs\HybridDualDistillation\lib\site-packages\shap\utils\_clustering.py:170: UserWarning: No/low signal found from feature 5 (this is typically caused by constant or near-constant features)! Cluster distances can't be computed for it (so setting all redundancy distances to 1).
  warnings.warn(
C:\Users\danie\anaconda3\envs\HybridDual

Saved SHAP summary plot for Traffic class ARP_Spoofing to D:\HybridDualDistillation\IoMTDataset\IoMT-Distillation\checkpoints\shap_summary_student_1d_cnn_no_kd_traffic_arp_spoofing_fold_single.pdf
Saved SHAP bar plot for Traffic class ARP_Spoofing to D:\HybridDualDistillation\IoMTDataset\IoMT-Distillation\checkpoints\shap_bar_student_1d_cnn_no_kd_traffic_arp_spoofing_fold_single.pdf
Saved SHAP beeswarm plot for Traffic class ARP_Spoofing to D:\HybridDualDistillation\IoMTDataset\IoMT-Distillation\checkpoints\shap_beeswarm_student_1d_cnn_no_kd_traffic_arp_spoofing_fold_single.pdf
Saved SHAP local bar plot for Traffic class ARP_Spoofing to D:\HybridDualDistillation\IoMTDataset\IoMT-Distillation\checkpoints\shap_local_bar_student_1d_cnn_no_kd_traffic_arp_spoofing_fold_single.pdf
Saved SHAP clustered bar plot for Traffic class ARP_Spoofing to D:\HybridDualDistillation\IoMTDataset\IoMT-Distillation\checkpoints\shap_clustered_bar_student_1d_cnn_no_kd_traffic_arp_spoofing_fold_single.pdf
Saved